# S6E2 LightGBM & XGBoost Tuning
Baseline (proba submissions): LGBM 0.95326, XGB 0.95308, CatBoost 0.95356

Strategy: early stopping to find optimal iterations, then focused grid on the
most impactful params for each architecture.

- **LGBM**: `num_leaves` (leaf-wise growth driver) × `min_child_samples` (regularization)
- **XGB**: `max_depth` × `min_child_weight` × `gamma` (pruning)

## Imports & Data

In [1]:
import numpy as np
import pandas as pd
import subprocess
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb

KAGGLE_DATA = Path("/kaggle/input/playground-series-s6e2")
LOCAL_DATA  = Path("data")
DATA_DIR    = KAGGLE_DATA if KAGGLE_DATA.exists() else LOCAL_DATA

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
    if "heart_disease" in df.columns:
        df["heart_disease"] = df["heart_disease"].map({"Absence": 0, "Presence": 1})
    return df

train = prep(pd.read_csv(DATA_DIR / "train.csv"))
test  = prep(pd.read_csv(DATA_DIR / "test.csv"))
ss    = pd.read_csv(DATA_DIR / "sample_submission.csv")

FEATURES = [c for c in train.columns if c not in ["heart_disease", "id"]]
X = train[FEATURES]; y = train["heart_disease"]; X_test = test[FEATURES]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Train: {X.shape}  Test: {X_test.shape}")

Train: (630000, 13)  Test: (270000, 13)


## Submit Helper

In [2]:
def submit(model, name, cv_auc):
    model.fit(X, y)
    sub = ss.copy()
    sub["Heart Disease"] = model.predict_proba(X_test)[:, 1]
    fname = f"submissions/{name}.csv"
    desc  = f"{name} | cv_auc={cv_auc:.4f}"
    sub.to_csv(fname, index=False)
    r = subprocess.run(
        ["kaggle", "competitions", "submit", "-c", "playground-series-s6e2",
         "-f", fname, "-m", desc],
        capture_output=True, text=True
    )
    status = "submitted" if "successfully" in r.stdout.lower() else r.stdout.strip()[:80]
    print(f"{name}: {status}  |  desc: {desc}")

---
# LightGBM Tuning

## LGBM 1. Early Stopping
Find optimal n_estimators at lr=0.05 (max 2000).

In [3]:
lgb_es_params = dict(
    n_estimators=2000, learning_rate=0.05, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8,
    device="gpu", random_state=42, n_jobs=-1, verbose=-1
)

lgb_best_iters, lgb_aucs = [], []
for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    m = lgb.LGBMClassifier(**lgb_es_params)
    m.fit(X_tr, y_tr,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    lgb_best_iters.append(m.best_iteration_)
    lgb_aucs.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    print(f"  fold {fold+1}: best_iter={m.best_iteration_}  auc={lgb_aucs[-1]:.4f}")

lgb_n = int(np.mean(lgb_best_iters))
print(f"\nMean best_iter: {lgb_n}")
print(f"ES lr=0.05   cv_auc={np.mean(lgb_aucs):.4f}")
print(f"Default      cv_auc=0.9551")

  fold 1: best_iter=632  auc=0.9556
  fold 2: best_iter=536  auc=0.9546
  fold 3: best_iter=545  auc=0.9554
  fold 4: best_iter=560  auc=0.9549
  fold 5: best_iter=676  auc=0.9557

Mean best_iter: 589
ES lr=0.05   cv_auc=0.9552
Default      cv_auc=0.9551


## LGBM 2. Grid: num_leaves × min_child_samples
`num_leaves` controls leaf-wise tree complexity (LGBM's equivalent of depth).
`min_child_samples` prevents overfitting on small leaves.

In [4]:
leaves_grid = [31, 63, 127, 255]
mcs_grid    = [20, 50, 100, 200]

lgb_grid = []
for nl in leaves_grid:
    for mcs in mcs_grid:
        params = dict(
            n_estimators=lgb_n, learning_rate=0.05, num_leaves=nl,
            min_child_samples=mcs,
            subsample=0.8, colsample_bytree=0.8,
            device="gpu", random_state=42, n_jobs=-1, verbose=-1
        )
        auc = cross_val_score(
            lgb.LGBMClassifier(**params), X, y, cv=cv, scoring="roc_auc"
        ).mean()
        lgb_grid.append({"num_leaves": nl, "min_child_samples": mcs, "cv_auc": auc})
        print(f"num_leaves={nl:3d}  mcs={mcs:3d}  auc={auc:.4f}")

lgb_grid_df = pd.DataFrame(lgb_grid).sort_values("cv_auc", ascending=False)
print("\nTop 5:")
print(lgb_grid_df.head().to_string(index=False))

num_leaves= 31  mcs= 20  auc=0.9552
num_leaves= 31  mcs= 50  auc=0.9552
num_leaves= 31  mcs=100  auc=0.9553
num_leaves= 31  mcs=200  auc=0.9553
num_leaves= 63  mcs= 20  auc=0.9551
num_leaves= 63  mcs= 50  auc=0.9551
num_leaves= 63  mcs=100  auc=0.9551
num_leaves= 63  mcs=200  auc=0.9551
num_leaves=127  mcs= 20  auc=0.9548


KeyboardInterrupt: 

## LGBM 3. Submit Best

In [ ]:
lgb_best = lgb_grid_df.iloc[0]
lgb_best_nl  = int(lgb_best["num_leaves"])
lgb_best_mcs = int(lgb_best["min_child_samples"])
lgb_best_auc = lgb_best["cv_auc"]

print(f"Best: num_leaves={lgb_best_nl}  min_child_samples={lgb_best_mcs}  cv_auc={lgb_best_auc:.4f}")
print(f"vs default cv_auc=0.9551  delta={lgb_best_auc - 0.9551:+.4f}")

lgb_tuned = lgb.LGBMClassifier(
    n_estimators=lgb_n, learning_rate=0.05,
    num_leaves=lgb_best_nl, min_child_samples=lgb_best_mcs,
    subsample=0.8, colsample_bytree=0.8,
    device="gpu", random_state=42, n_jobs=-1, verbose=-1
)
submit(lgb_tuned, "lgbm_tuned_v1", lgb_best_auc)

---
# XGBoost Tuning

## XGB 1. Early Stopping
Find optimal n_estimators at lr=0.05 (max 2000).

In [5]:
xgb_es_params = dict(
    n_estimators=2000, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="auc", early_stopping_rounds=50,
    device="cuda", random_state=42, n_jobs=-1
)

xgb_best_iters, xgb_aucs = [], []
for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    m = xgb.XGBClassifier(**xgb_es_params, verbosity=0)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    xgb_best_iters.append(m.best_iteration)
    xgb_aucs.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    print(f"  fold {fold+1}: best_iter={m.best_iteration}  auc={xgb_aucs[-1]:.4f}")

xgb_n = int(np.mean(xgb_best_iters))
print(f"\nMean best_iter: {xgb_n}")
print(f"ES lr=0.05   cv_auc={np.mean(xgb_aucs):.4f}")
print(f"Default      cv_auc=0.9549")

  fold 1: best_iter=488  auc=0.9556
  fold 2: best_iter=416  auc=0.9546
  fold 3: best_iter=428  auc=0.9554
  fold 4: best_iter=448  auc=0.9550
  fold 5: best_iter=528  auc=0.9558

Mean best_iter: 461
ES lr=0.05   cv_auc=0.9553
Default      cv_auc=0.9549


I## XGB 2. Grid: max_depth × min_child_weight
`min_child_weight` is XGB's main regularization knob (covers instance weight in a leaf).

In [ ]:
depth_grid = [5, 6, 7, 8]
mcw_grid   = [1, 3, 5, 10]

xgb_grid = []
for depth in depth_grid:
    for mcw in mcw_grid:
        params = dict(
            n_estimators=xgb_n, learning_rate=0.05, max_depth=depth,
            min_child_weight=mcw,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="logloss", device="cuda", random_state=42, n_jobs=-1
        )
        auc = cross_val_score(
            xgb.XGBClassifier(**params, verbosity=0), X, y, cv=cv, scoring="roc_auc"
        ).mean()
        xgb_grid.append({"max_depth": depth, "min_child_weight": mcw, "cv_auc": auc})
        print(f"depth={depth}  mcw={mcw:2d}  auc={auc:.4f}")

xgb_grid_df = pd.DataFrame(xgb_grid).sort_values("cv_auc", ascending=False)
print("\nTop 5:")
print(xgb_grid_df.head().to_string(index=False))

## XGB 3. Submit Best

In [ ]:
xgb_best = xgb_grid_df.iloc[0]
xgb_best_depth = int(xgb_best["max_depth"])
xgb_best_mcw   = int(xgb_best["min_child_weight"])
xgb_best_auc   = xgb_best["cv_auc"]

print(f"Best: max_depth={xgb_best_depth}  min_child_weight={xgb_best_mcw}  cv_auc={xgb_best_auc:.4f}")
print(f"vs default cv_auc=0.9549  delta={xgb_best_auc - 0.9549:+.4f}")

xgb_tuned = xgb.XGBClassifier(
    n_estimators=xgb_n, learning_rate=0.05,
    max_depth=xgb_best_depth, min_child_weight=xgb_best_mcw,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", device="cuda", random_state=42, n_jobs=-1, verbosity=0
)
submit(xgb_tuned, "xgb_tuned_v1", xgb_best_auc)

## Summary

In [ ]:
print("Model                    CV AUC   LB AUC")
print("-" * 45)
print(f"xgb_default_proba        0.9549   0.95308")
print(f"lgbm_default_proba       0.9551   0.95326")
print(f"catboost_default_proba   0.9554   0.95356")
print(f"lgbm_tuned_v1            {lgb_best_auc:.4f}   (pending)")
print(f"xgb_tuned_v1             {xgb_best_auc:.4f}   (pending)")